In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

df = pd.read_csv("pos_tags.csv")

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

df = df[['word', 'pos_tag']].dropna()

if len(df) > 5000:
    df = df.sample(n=5000, random_state=42)

train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)

print("Dataset Loaded")
print("Training Samples :", len(train_df))
print("Testing Samples  :", len(test_df))

words = sorted(train_df["word"].unique())
tags = sorted(train_df["pos_tag"].unique())

word_to_idx = {w: i for i, w in enumerate(words)}
tag_to_idx = {t: i for i, t in enumerate(tags)}
idx_to_tag = {i: t for t, i in tag_to_idx.items()}

num_words = len(words)
num_tags = len(tags)

print("Building Initial Matrix...")

initial = np.ones(num_tags)

for tag in train_df["pos_tag"]:
    initial[tag_to_idx[tag]] += 1

initial /= initial.sum()

print("Building Transition Matrix...")

transition = np.ones((num_tags, num_tags))
transition /= transition.sum(axis=1, keepdims=True)

print("Building Emission Matrix...")

emission = np.ones((num_tags, num_words))

for _, row in train_df.iterrows():
    emission[tag_to_idx[row["pos_tag"]], word_to_idx[row["word"]]] += 1

emission /= emission.sum(axis=1, keepdims=True)

log_initial = np.log(initial)
log_emission = np.log(emission)

print("Predicting...")

def predict_word(word):
    if word in word_to_idx:
        score = log_initial + log_emission[:, word_to_idx[word]]
        return idx_to_tag[np.argmax(score)]
    return idx_to_tag[np.argmax(log_initial)]

y_true = test_df["pos_tag"].tolist()
y_pred = [predict_word(word) for word in test_df["word"]]

print("\nAccuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, average="weighted", zero_division=0))
print("Recall   :", recall_score(y_true, y_pred, average="weighted", zero_division=0))
print("F1 Score :", f1_score(y_true, y_pred, average="weighted", zero_division=0))

print("\nClassification Report\n")
print(classification_report(y_true, y_pred, zero_division=0))

print("\nTesting on Unseen Sentences\n")

sentences = [
    ["the", "cat", "runs"],
    ["she", "is", "happy"],
    ["computer", "science"],
    ["artificial", "intelligence"],
    ["python", "programming"]
]

for sentence in sentences:
    predicted = [predict_word(word) for word in sentence]
    print("Sentence :", " ".join(sentence))
    print("Tags     :", predicted)
    print()

Dataset Loaded
Training Samples : 4500
Testing Samples  : 500
Building Initial Matrix...
Building Transition Matrix...
Building Emission Matrix...
Predicting...

Accuracy : 0.61
Precision: 0.3721
Recall   : 0.61
F1 Score : 0.4622360248447205

Classification Report

              precision    recall  f1-score   support

          JJ       0.00      0.00      0.00        68
          NN       0.61      1.00      0.76       305
         NNS       0.00      0.00      0.00        63
          RB       0.00      0.00      0.00        31
          VB       0.00      0.00      0.00         1
         VBD       0.00      0.00      0.00         1
         VBG       0.00      0.00      0.00        19
         VBN       0.00      0.00      0.00        12

    accuracy                           0.61       500
   macro avg       0.08      0.12      0.09       500
weighted avg       0.37      0.61      0.46       500


Testing on Unseen Sentences

Sentence : the cat runs
Tags     : ['NN', 'NN', 'NN']